In [1]:
import os
import re
from datetime import datetime

import pandas as pd
import numpy as np
import yfinance as yf
from tqdm import tqdm

In [2]:
# Configuration
START_DATE = "2003-01-02"
END_DATE = "2026-03-15"

print(f"Downloading daily data from {START_DATE} to {END_DATE}")

In [3]:
# Setup output directory
BASE = os.path.abspath(os.getcwd())
if os.path.basename(BASE) == "Data":
    BASE = os.path.dirname(BASE)

RAW_DIR = os.path.join(BASE, "Data", "Outputs", "Raw_Data")
os.makedirs(RAW_DIR, exist_ok=True)
print(f"Output directory: {RAW_DIR}")

Output directory: /Users/kamilkashif/Documents/University/Masters Thesis/Master_Thesis_DRL_EUROSTOXX/Data/Outputs/Raw_Data


## Extract all tickers: t0 baseline (`eustx_membership_2003-01-02.csv`) ∪ `EUSTX_changes`

In [4]:
# Bloomberg composite exchange code -> Yahoo suffix (Euro Stoxx 50 listings)
BBG_EXCH_TO_YAHOO = {
    "NA": ".AS",
    "FP": ".PA",
    "GY": ".DE",
    "IM": ".MI",
    "SQ": ".MC",
    "SM": ".MC",
    "ID": ".IR",
    "FH": ".HE",
    "BB": ".BR",
}


def normalize_ticker(bloomberg_ticker):
    """Bloomberg 'SYMBOL EX Equity' -> Yahoo Finance symbol (European listings)."""
    if pd.isna(bloomberg_ticker) or not bloomberg_ticker:
        return ""
    parts = str(bloomberg_ticker).strip().split()
    if len(parts) < 2:
        return parts[0] if parts else ""
    symbol, exch = parts[0], parts[1].upper()
    if re.match(r"^\d{7,8}[A-Z]$", symbol.upper()):
        return symbol
    suf = BBG_EXCH_TO_YAHOO.get(exch)
    if suf:
        return f"{symbol}{suf}"
    return symbol


def extract_all_tickers_from_eustx_log(filepath):
    """Parse EUSTX_changes and extract all unique yfinance symbols (ADD/DEL history only)."""
    if not os.path.isfile(filepath):
        raise FileNotFoundError(f"EUSTX_changes not found: {filepath}")
    
    tickers = set()
    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        in_add = False
        in_del = False
        
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            # Detect section headers
            if "ADD" in line.upper() and "DEL" not in line.upper():
                in_add = True
                in_del = False
                continue
            if "DEL" in line.upper() and "ADD" not in line.upper():
                in_del = True
                in_add = False
                continue
            
            # Skip metadata lines
            if "TOTAL" in line.upper() or "PERIOD" in line.upper():
                continue
            if re.match(r"^-+$", line):  # separator lines
                continue
            if "!!!" in line or "[END" in line:
                continue
            
            # Extract tickers from ADD/DEL sections
            if in_add or in_del:
                # Skip words that are not tickers
                skip_words = ("ADD", "DEL", "PERIOD", "TOTAL", "Equity", "COUNT", 
                              "File", "Edit", "Format", "View", "Help", "LEND")
                
                # Split by common separators
                parts = re.split(r"[+*•\-]", line)
                for part in parts:
                    part = part.strip()
                    if not part:
                        continue
                    
                    part_clean = part.strip().lstrip("+*•").strip() if part.strip() else ""
                    if part_clean.startswith("-"):
                        part_clean = part_clean.lstrip("-").strip()
                    tokens = part_clean.split()
                    if not tokens:
                        continue
                    if tokens[0] in skip_words:
                        continue
                    ticker = normalize_ticker(part_clean)
                    if ticker and re.match(r"^[A-Z0-9\.\-]{2,16}$", ticker.upper()):
                        tickers.add(ticker)
    
    return sorted(tickers)


# EUSTX_changes (rebalances) + t0 baseline CSV (Euro Stoxx 50 at 2003-01-02)
DATA_DIR = os.path.join(BASE, "Data")
EUSTX_LOG = os.path.join(DATA_DIR, "EUSTX_changes")
if not os.path.isfile(EUSTX_LOG):
    EUSTX_LOG = os.path.join(BASE, "EUSTX_changes")

BASELINE_CSV = os.path.join(DATA_DIR, "eustx_membership_2003-01-02.csv")

# Tickers from rebalance log
from_log = set(extract_all_tickers_from_eustx_log(EUSTX_LOG))

# Tickers from t0 snapshot
df_b = pd.read_csv(BASELINE_CSV)
col = "ticker_bloomberg" if "ticker_bloomberg" in df_b.columns else df_b.columns[0]
from_baseline = {normalize_ticker(str(x).strip()) for x in df_b[col].astype(str)}
from_baseline = {t for t in from_baseline if t}

ALL_TICKERS = sorted(from_log | from_baseline)
print(f"\nUnique tickers: {len(ALL_TICKERS)} (from EUSTX_changes: {len(from_log)}, t0 CSV: {len(from_baseline)})")
print(f"Sample: {ALL_TICKERS[:20]}")

# Identify Bloomberg-only IDs (numeric prefixes that won't work on yfinance)
bloomberg_only = [t for t in ALL_TICKERS if re.match(r"^\d{7,8}[A-Z]$", t)]
yfinance_compatible = [t for t in ALL_TICKERS if t not in bloomberg_only]

print(f"\nBloomberg-only IDs (will fail on yfinance): {len(bloomberg_only)}")
print(f"  {bloomberg_only}")
print(f"\nyfinance-compatible tickers: {len(yfinance_compatible)}")


Unique tickers: 102 (from EUSTX_changes: 84, t0 CSV: 50)
Sample: ['1844030D', '3577044Z', '63DU.DE', 'ABI.BR', 'ACA.PA', 'AD.AS', 'ADS.DE', 'ADYEN.AS', 'AGN.AS', 'AGS.BR', 'AI.PA', 'AIBG.IR', 'AIR.PA', 'ALO.PA', 'ALU.PA', 'ALV.DE', 'AMS.MC', 'ARGX.BR', 'ASML.AS', 'AVE.PA']

Bloomberg-only IDs (will fail on yfinance): 2
  ['1844030D', '3577044Z']

yfinance-compatible tickers: 100


## Download Data from yfinance

In [5]:
def download_ticker_data(ticker, start_date, end_date):
    """Download daily OHLCV data for a single ticker from yfinance.
    
    Returns:
        DataFrame with columns: date, open, high, low, close, volume
        None if download fails or data is empty
    """
    try:
        # Download data
        data = yf.download(
            ticker,
            start=start_date,
            end=end_date,
            interval="1d",
            progress=False
        )
        
        if data.empty:
            return None
        
        # Handle MultiIndex columns (yfinance returns MultiIndex even for single ticker)
        if isinstance(data.columns, pd.MultiIndex):
            # Flatten MultiIndex: ('Close', 'AAPL') -> 'Close'
            data.columns = data.columns.get_level_values(0)
        
        # Reset index to get date as a column
        data = data.reset_index()
        
        # Normalize column names to lowercase
        data.columns = [c.lower() for c in data.columns]
        
        # Select OHLCV columns (in correct order)
        required_cols = ['date', 'open', 'high', 'low', 'close', 'volume']
        available_cols = [c for c in required_cols if c in data.columns]
        
        if 'date' not in available_cols or 'close' not in available_cols:
            return None
        
        result = data[available_cols]
        
        # Ensure volume column exists (some tickers might not have it)
        if 'volume' not in result.columns:
            result['volume'] = 0
        
        return result
    
    except Exception as e:
        # Optionally print error for debugging
        # print(f"Error downloading {ticker}: {e}")
        return None

In [6]:
# Download data for all tickers
tickers_retrieved = []
tickers_not_retrieved = []
ticker_info = {}  # Track first/last date and row count

print(f"\nDownloading {len(ALL_TICKERS)} tickers...\n")

for ticker in tqdm(ALL_TICKERS, desc="Downloading"):
    df = download_ticker_data(ticker, START_DATE, END_DATE)
    
    if df is not None and len(df) > 0:
        # Save to CSV
        output_path = os.path.join(RAW_DIR, f"{ticker}.csv")
        df.to_csv(output_path, index=False)
        
        # Track info
        tickers_retrieved.append(ticker)
        ticker_info[ticker] = {
            'first_date': df['date'].min(),
            'last_date': df['date'].max(),
            'row_count': len(df)
        }
    else:
        tickers_not_retrieved.append(ticker)

print(f"\n{'='*60}")
print(f"DOWNLOAD SUMMARY")
print(f"{'='*60}")
print(f"Total tickers attempted: {len(ALL_TICKERS)}")
print(f"Successfully downloaded: {len(tickers_retrieved)}")
print(f"Failed/unavailable: {len(tickers_not_retrieved)}")
print(f"\nSuccess rate: {len(tickers_retrieved)/len(ALL_TICKERS)*100:.1f}%")

Downloading:   0%|          | 0/102 [00:00<?, ?it/s]HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 1844030D"}}}
$1844030D: possibly delisted; no timezone found

1 Failed download:
['1844030D']: possibly delisted; no timezone found
Downloading:   1%|          | 1/102 [00:01<02:35,  1.53s/it]HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 3577044Z"}}}
$3577044Z: possibly delisted; no timezone found

1 Failed download:
['3577044Z']: possibly delisted; no timezone found
Downloading:   2%|▏         | 2/102 [00:02<02:06,  1.27s/it]$63DU.DE: possibly delisted; no price data found  (1d 2003-01-02 -> 2026-03-15)

1 Failed download:
['63DU.DE']: possibly delisted; no price data found  (1d 2003-01-02 -> 2026-03-15)
Downloading:  11%|█         | 11/102 [00:07<00:49,  1.82it/s]$AIBG.IR: possibly delisted; no timezone found

1 Failed download:
['AIBG.IR']: poss


DOWNLOAD SUMMARY
Total tickers attempted: 102
Successfully downloaded: 79
Failed/unavailable: 23

Success rate: 77.5%


In [7]:
# Show failed tickers breakdown
failed_bloomberg_ids = [t for t in tickers_not_retrieved if t in bloomberg_only]
failed_regular = [t for t in tickers_not_retrieved if t not in bloomberg_only]

print(f"\nFailed tickers breakdown:")
print(f"  Bloomberg IDs (expected to fail): {len(failed_bloomberg_ids)}")
print(f"    {failed_bloomberg_ids}")
print(f"\n  Regular tickers (delisted/unavailable): {len(failed_regular)}")
print(f"    {failed_regular}")


Failed tickers breakdown:
  Bloomberg IDs (expected to fail): 2
    ['1844030D', '3577044Z']

  Regular tickers (delisted/unavailable): 21
    ['63DU.DE', 'AIBG.IR', 'ALU.PA', 'AVE.PA', 'CRH.IR', 'FLTR.IR', 'FORA.AS', 'HVM.DE', 'LG.PA', 'LINU.DE', 'MTP.PA', 'NDA.HE', 'RDA.AS', 'SPI.MI', 'SZE.PA', 'TI.MI', 'TIM.MI', 'UL.AS', 'UL.PA', 'UNAT.AS', 'URW.AS']


## Save Tracking Files

In [8]:
# Save tickers_retrieved.csv
df_retrieved = pd.DataFrame({
    'ticker': tickers_retrieved,
    'first_date': [ticker_info[t]['first_date'] for t in tickers_retrieved],
    'last_date': [ticker_info[t]['last_date'] for t in tickers_retrieved],
    'row_count': [ticker_info[t]['row_count'] for t in tickers_retrieved]
})
df_retrieved.to_csv(os.path.join(RAW_DIR, "tickers_retrieved.csv"), index=False)
print(f"Saved tickers_retrieved.csv ({len(tickers_retrieved)} tickers)")

# Save tickers_not_retrieved.csv
df_not_retrieved = pd.DataFrame({
    'ticker': tickers_not_retrieved,
    'reason': ['Bloomberg ID' if t in bloomberg_only else 'Unavailable/Delisted' 
               for t in tickers_not_retrieved]
})
df_not_retrieved.to_csv(os.path.join(RAW_DIR, "tickers_not_retrieved.csv"), index=False)
print(f"Saved tickers_not_retrieved.csv ({len(tickers_not_retrieved)} tickers)")

print(f"\nAll tracking files saved to: {RAW_DIR}")

Saved tickers_retrieved.csv (79 tickers)
Saved tickers_not_retrieved.csv (23 tickers)

All tracking files saved to: /Users/kamilkashif/Documents/University/Masters Thesis/Master_Thesis_DRL_EUROSTOXX/Data/Outputs/Raw_Data


## Data Quality Check

In [9]:
# Show data coverage statistics
print("\nData Coverage Statistics:")
print(f"Earliest data: {df_retrieved['first_date'].min()}")
print(f"Latest data: {df_retrieved['last_date'].max()}")
print(f"\nRow count distribution:")
print(df_retrieved['row_count'].describe())

# Show tickers with limited data
limited_data = df_retrieved[df_retrieved['row_count'] < 1000].sort_values('row_count')
if len(limited_data) > 0:
    print(f"\nTickers with <1000 rows (recently listed or delisted):")
    print(limited_data[['ticker', 'first_date', 'last_date', 'row_count']].head(20))


Data Coverage Statistics:
Earliest data: 2003-01-02 00:00:00
Latest data: 2026-03-13 00:00:00

Row count distribution:
count      79.000000
mean     5522.810127
std      1123.650918
min      1148.000000
25%      5910.000000
50%      5912.000000
75%      5949.000000
max      5954.000000
Name: row_count, dtype: float64


## Sample Random Ticker

In [10]:
# Display a random ticker's data
if len(tickers_retrieved) > 0:
    random_ticker = tickers_retrieved[np.random.randint(0, len(tickers_retrieved))]
    df_sample = pd.read_csv(os.path.join(RAW_DIR, f"{random_ticker}.csv"))
    print(f"\nSample data for {random_ticker}:")
    print(f"Shape: {df_sample.shape}")
    print(f"\nFirst 5 rows:")
    print(df_sample.head())
    print(f"\nLast 5 rows:")
    print(df_sample.tail())
    print(f"\nColumn types:")
    print(df_sample.dtypes)


Sample data for TEF.MC:
Shape: (5944, 6)

First 5 rows:
         date      open      high       low     close    volume
0  2003-01-02  2.306193  2.306193  2.168586  2.306193  21956492
1  2003-01-03  2.296001  2.326580  2.257776  2.296001  20292601
2  2003-01-06  2.296001  2.296001  2.296001  2.296001         0
3  2003-01-07  2.428510  2.431058  2.349514  2.428510  41805519
4  2003-01-08  2.443801  2.497315  2.385191  2.443801  49548037

Last 5 rows:
            date   open   high    low  close    volume
5939  2026-03-09  3.600  3.606  3.534  3.583  12286718
5940  2026-03-10  3.600  3.649  3.589  3.615   9144316
5941  2026-03-11  3.580  3.619  3.504  3.514  14990438
5942  2026-03-12  3.514  3.550  3.475  3.549  10191128
5943  2026-03-13  3.519  3.630  3.492  3.589  11860032

Column types:
date          str
open      float64
high      float64
low       float64
close     float64
volume      int64
dtype: object
